In [1]:
import datetime
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import RunReportRequest
import os
import pandas as pd

In [2]:
date1 = datetime.datetime.today().strftime('%Y-%m-%d')
date2 = (datetime.datetime.today() - datetime.timedelta(days=500)).strftime('%Y-%m-%d')


In [3]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import RunReportRequest
import os

# Set path to your service account JSON key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "cred.json"

# Initialize client
client = BetaAnalyticsDataClient()

# Replace with your GA4 property ID
property_id = "459674973"

# Request example: Sessions by date
request = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[{"name": "date"}],
    metrics=[{"name": "sessions"}],
    date_ranges=[{"start_date": date2, "end_date": date1}],
)

response = client.run_report(request)

date = [d.dimension_values[0].value for d in response.rows]
val = [d.metric_values[0].value for d in response.rows]

import pandas as pd
import datetime
df = pd.DataFrame({
    'Date': date,
    'value': val
})
df['Date'] = [str(z)[:4] + '.' + str(z)[4:6] + '.' + str(z)[6:] for z in df['Date']]
df['Date'] = [datetime.datetime.strptime(z, '%Y.%m.%d') for z in df['Date']]
df['value'] = [int(z) for z in df['value']]
df = df.sort_values('Date')
df = df[df['value'] > 10]

In [118]:
sayfalar = ['/tr','/tr/haberler','/tr/blog','/tr/yayin','/tr/video','/tr/podcast','/tr/ekibimiz','/tr/arastirmacilar','/en','/en/haberler','/en/blog','/en/yayin','/en/video','/en/podcast','/en/ekibimiz','/en/arastirmacilar']

In [182]:
sayfalar = ['/tr/blog/s/7610']

In [183]:
datum = pd.DataFrame()
for sayfa in sayfalar:
    # Set path to your service account JSON key
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "cred.json"

    # Initialize client
    client = BetaAnalyticsDataClient()

    # Replace with your GA4 property ID
    property_id = "459674973"


    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[{"name": "date"}],
        metrics=[{"name": "sessions"}],
        date_ranges=[{"start_date": date2, "end_date": date1}],
        dimension_filter={"filter": {"field_name": "pagePath", "string_filter": {"value": sayfa}}}
    )

    response = client.run_report(request)

    date = [d.dimension_values[0].value for d in response.rows]
    val = [d.metric_values[0].value for d in response.rows]

    import pandas as pd
    import datetime
    df = pd.DataFrame({
        'Date': date,
        'value': val
    })
    df['tür'] = sayfa.replace('/','_')[1:]
    df['Date'] = [str(z)[:4] + '.' + str(z)[4:6] + '.' + str(z)[6:] for z in df['Date']]
    df['Date'] = [datetime.datetime.strptime(z, '%Y.%m.%d') for z in df['Date']]
    df['value'] = [int(z) for z in df['value']]
    df = df.sort_values('Date')

    datum = pd.concat([datum,df])


In [184]:
df2 = datum.groupby(['Date']).sum('value').reset_index()

In [191]:
df2 = datum.groupby(['Date']).sum('value').reset_index()
df2['value'] = df2['value'].rolling(30).sum()
df2 = df2.dropna()

In [192]:
df['value'] = df['value'].rolling(30).sum()
df = df.dropna()

In [193]:
a = (df.set_index('Date') - df2.set_index('Date')).reset_index()

In [194]:
import plotly.express as px

px.line(df2, x='Date',y='value')

In [ ]:
import plotly.express as px

px.line(df2, x='Date',y='value')